# 9 · Discontinuous Galerkin & HDG

Different motivations exist to leave conforming settings (e.g. some stabilizations for convection are somewhat more generic if we leave the continuous ($H^1$) elements):
* **Discontinuous Galerkin (DG)** breaks the continuity
* each element carries its own polynomial,

This adds more unknowns and more couplings.

**Hybrid DG (HDG)**:
* adds more unknowns on the **facets**
* and **statically condenses** the element interiors away,
shrinking the global system $\leadsto$ more efficient DG (when it comes to solving linear systems).

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from ngsolve import *
from netgen.occ import WorkPlane, OCCGeometry, X, Y
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. A convection-dominated problem — where $H^1$ wobbles

Same double-glazing wind as unit 8, but now with **tiny diffusion** $\varepsilon=10^{-3}$:
$-\varepsilon\Delta u + \mathbf b\!\cdot\!\nabla u = 0$, hot right wall ($u=1$), cold left
($u=0$).
Standard continuous `H1` (no stabilzation) $\leadsto$ oscillations.

In [ ]:
square = WorkPlane().MoveTo(-1, -1).Rectangle(2, 2).Face()
square.edges.Min(X).name = "left";   square.edges.Max(X).name = "right"
square.edges.Min(Y).name = "bottom"; square.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(square, dim=2).GenerateMesh(maxh=0.18))

eps, order = 1e-3, 2
wind = CF((2*y*(1-x*x), -2*x*(1-y*y)))

u, v = H1(mesh, order=order, dirichlet="right|left").TnT()
uH1 = Solve(eps*grad(u)*grad(v)*dx + (wind*grad(u))*v*dx == 0, u[BND("right")]==CF(1))
Draw(uH1,mesh, "uH1", deformation = True)

## 2. Upwind DG — broken elements, fluxes across edges

We use a **broken** space `L2(..., dgjumps=True)` (no continuity). Two ingredients on the
**interior edges** (`dx(skeleton=True)`):

* **convection** by an **upwind** flux — the edge value is taken from whichever side the wind
  blows *from* using two new `CoefficientFunction`s:
  * `IfPos` (case switch for `CF`s)
  * `...Other()` The trace value of the FE, but from the neighbor (or "slave" element of a facet)
```
   IfPos(b*n, u, u.Other())
```
* **diffusion** by the **symmetric interior penalty** (consistency + symmetry + an $\alpha/h$ penalty on the jump).

Dirichlet walls are imposed **weakly** (Nitsche).

In [ ]:
n = specialcf.normal(2); h = specialcf.mesh_size; alpha = 4*order*order
ud = IfPos(x, 1.0, 0.0) # right wall (x>0) hot=1, left wall cold=0

We need different integral types for DG:

In [ ]:
dS = dx(skeleton=True)                                          # integrate over interior edges
dD = ds(skeleton=True, definedon=mesh.Boundaries("right|left")) # integrate over Dirichlet edges (but from vol. el. point of view)

In [ ]:
V2 = L2(mesh, order=order, dgjumps=True)
u, v = V2.TnT()
jmp = lambda w: w - w.Other()
mn = lambda gw: 0.5*(gw + gw.Other())*n
bn = wind*n
a2 = BilinearForm(V2)
a2 += eps*grad(u)*grad(v)*dx                                                       # diffusion (volume)
a2 += eps*(-mn(grad(u))*jmp(v) - mn(grad(v))*jmp(u) + alpha/h*jmp(u)*jmp(v))*dS    # SIP (interior)
a2 += eps*(-grad(u)*n*v - grad(v)*n*u + alpha/h*u*v)*dD                            # SIP Nitsche (Dirichlet)
a2 += -u*(wind*grad(v))*dx                                                         # convection (volume, IBP)
a2 += bn*IfPos(bn, u, u.Other())*jmp(v)*dS                                         # convection (upwind edge)
a2.Assemble()

f2 = LinearForm(eps*(-grad(v)*n*ud + alpha/h*ud*v)*dD).Assemble()                  # Dirichlet data -> rhs

uDG = GridFunction(V2)
uDG.vec.data = a2.mat.Inverse(V2.FreeDofs(), inverse="umfpack") * f2.vec
Draw(uDG, mesh, "u (DG)", min=0, max=1, autoscale=False, deformation = True)

The contrast is stark — $H^1$ rings, DG does not:

In [ ]:
#matplotlib plot using point values...
gx = gy = np.linspace(-1, 1, 400)
def sample(gf):
    return np.array([[gf(mesh(float(xx), float(yy))) for xx in gx] for yy in gy])
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
c0 = ax[0].contourf(gx, gy, sample(uH1), levels=np.linspace(0, 1, 21), cmap="jet")
c1 = ax[1].contourf(gx, gy, sample(uDG), levels=np.linspace(0, 1, 21), cmap="jet")
ax[0].set_title("Std. FEM (k=2; no stab.)"); fig.colorbar(c0, ax=ax[0])
ax[1].set_title("upwind DG (k=2"); fig.colorbar(c1, ax=ax[1])
for a in ax: a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 3. HDG — a facet unknown, and static condensation

DG couples every element to its neighbours, so **all** its dofs are global.

* **Hybrid DG** introduces a separate unknown $\hat u$ on the **facets** 
* and lets elements talk **only** to $\hat u$ (never directly to each other).

The element-interior dofs then couple **only within their element**, so they can be **eliminated locally** — *static condensation* (`condense=True`) — leaving a much smaller global system in the **facet** unknowns alone.

In [ ]:
Vl = L2(mesh, order=order)
Vf = FacetFESpace(mesh, order=order, dirichlet="right|left")  # <- polynomials on the skeleton/facets
X = Vl * Vf
(u, uhat), (v, vhat) = X.TnT()

dSb = dx(element_boundary=True)                         # integrate over each element's boundary

a3 = BilinearForm(X, condense=True)                    # <-- eliminate element interiors locally
iHDG = eps*grad(u)*grad(v)*dx
iHDG += eps*(-grad(u)*n*(v-vhat) - grad(v)*n*(u-uhat) + alpha/h*(u-uhat)*(v-vhat))*dSb   # HDG diffusion
iHDG += -u*(wind*grad(v))*dx                                                             # convection (volume)
iHDG += bn*IfPos(bn, u, uhat)*(v-vhat)*dSb                                               # upwind via facet
a3 += iHDG
a3.Assemble()

uHDG = GridFunction(X)
uHDG.components[1].Set(ud, definedon=mesh.Boundaries("right|left"))
res = (-a3.mat * uHDG.vec).Evaluate()                  # condensed solve (Dirichlet kept):

# the harmonic extension procedure:
res.data += a3.harmonic_extension_trans * res          #  1) lift element interiors to facets
uHDG.vec.data += a3.mat.Inverse(X.FreeDofs(coupling=True), inverse="umfpack") * res   # 2) global facet solve
uHDG.vec.data += a3.harmonic_extension * uHDG.vec      #  3) extend facets back into interiors
uHDG.vec.data += a3.inner_solve * res                  #  4) local interior solve

Draw(uHDG.components[0], mesh, "u (HDG)", min=0, max=1, autoscale=False)

Same clean solution as DG, but the **globally coupled** system is **smaller** — exactly why HDG
scales well for the 3-D flow problems in Part III.

## 4. Sparsity — who couples to whom

Assemble the three formulations on a coarse mesh and look at **who couples to whom**:

- **CG** — neighbours share vertex/edge dofs: a compact, globally coupled matrix.
- **DG** — *every* dof is global; elements couple to their face-neighbours — a bigger,
  block-structured matrix.
- **HDG** — elements talk only through the **facets** (green); the **element interiors** (tan)
  couple only within their element and are **condensed away**, leaving the small green facet system.

> **When to reach for which**
>
> - **HDG** is great for **implicit linear systems**: static condensation leaves a small,
>   facet-only global solve
> - For **explicit operator applications**: no gain with HDG 

In [ ]:
# sparsity comparison again ... 
import matplotlib.pyplot as plt, numpy as np
msparse = Mesh(unit_square.GenerateMesh(maxh=0.4)); p = 2      # coarse so the patterns stay legible
def nz(mat):
    r, c, _ = mat.COO(); return np.array(r), np.array(c)

Vc = H1(msparse, order=p); tu, tv = Vc.TnT()
Ac = BilinearForm(grad(tu)*grad(tv)*dx).Assemble().mat
Vd = L2(msparse, order=p, dgjumps=True); tu, tv = Vd.TnT()
Ad = BilinearForm(grad(tu)*grad(tv)*dx + (tu-tu.Other())*(tv-tv.Other())*dx(skeleton=True)).Assemble().mat
Vl = L2(msparse, order=p); Vf = FacetFESpace(msparse, order=p); Xs = Vl*Vf
(tu, tuh), (tv, tvh) = Xs.TnT()
Ah = BilinearForm(grad(tu)*grad(tv)*dx + (tu-tuh)*(tv-tvh)*dx(element_boundary=True)).Assemble().mat

fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a, A, ttl, col in [(ax[0], Ac, f"CG (H1) — {Vc.ndof} dofs", "#1f77b4"),
                       (ax[1], Ad, f"DG (L2) — {Vd.ndof} dofs", "#d62728")]:
    r, c = nz(A); a.scatter(c, r, s=7, marker="s", color=col, lw=0); a.set_title(ttl, fontweight="bold")
r, c = nz(Ah); nint = Vl.ndof; mask = (r < nint) & (c < nint)                    # interiors come first in Vl*Vf
ax[2].scatter(c[mask], r[mask], s=7, marker="s", color="#c9a36a", lw=0, label="interiors (condensed)")
ax[2].scatter(c[~mask], r[~mask], s=7, marker="s", color="#2ca02c", lw=0,
              label=f"facets — global ({sum(Xs.FreeDofs(coupling=True))} dofs)")
ax[2].legend(loc="center left", fontsize=8, framealpha=0.92)
ax[2].set_title(f"HDG (L2×Facet) — {Xs.ndof} dofs", fontweight="bold")
for a in ax: a.invert_yaxis(); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
fig.suptitle("Who couples to whom — CG vs DG vs HDG system matrices (order 2, coarse mesh)")
fig.tight_layout()

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("08-unsteady-doubleglazing", "8 · Unsteady problems — the double-glazing flow")
    _next = ("10-nonlinear-allencahn", "10 · Nonlinear problems — Allen–Cahn & Newton")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))